# 03 — Baselines and models

Baselines first, then XGBoost and Prophet, all judged on the same expanding-window backtest.

> On an exchange rate the random walk is the benchmark the literature has repeatedly found hard to beat. On a *managed* rate that sits still 30% of the time it is harder still. A model that does not beat it has not earned its complexity.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "notebook_connected"  # CDN reference, not an embedded copy of plotly.js per figure
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

from mru_forecast import viz
from mru_forecast.config import load_config
from mru_forecast.data.cleaning import load_processed

config = load_config()
frame = load_processed(config)
TARGETS = [t["name"] for t in config["targets"]]
print(f"{len(frame)} business days: {frame.index.min().date()} to {frame.index.max().date()}")
frame.tail(3).round(4)


2053 business days: 2018-11-01 to 2026-09-14


,BRENT_OIL,EUR_MRU,EUR_USD,GOLD,USD_MRU
date,,,,,
2026-09-10,104.715,46.0719,1.1611,4391.7000,39.6801
2026-09-11,106.885,46.0719,1.1611,4384.1499,39.6801
2026-09-14,106.360,46.2481,1.1553,4338.7000,40.0300


## 1. The backtest design

No `train_test_split`. Every split is chronological and has three blocks in time order:

```
|........ train ........|.. calibration ..|.. test ..|
```

The calibration block is the price of a conformal guarantee: the quantile must come from data the model was never fitted on. Splits are anchored at the **end** of the sample, so the evaluation covers the most recent period.

In [2]:
from mru_forecast.evaluation.backtesting import make_splits, splits_summary

splits = make_splits(pd.DatetimeIndex(frame.index), config)
splits_summary(splits)

,split,train_start,train_end,n_train,calib_start,calib_end,n_calib,test_start,test_end,n_test
0,0,2018-11-01,2024-08-29,1521,2024-08-30,2025-08-14,250,2025-08-15,2025-09-12,21
1,1,2018-11-01,2024-09-27,1542,2024-09-30,2025-09-12,250,2025-09-15,2025-10-13,21
2,2,2018-11-01,2024-10-28,1563,2024-10-29,2025-10-13,250,2025-10-14,2025-11-11,21
3,3,2018-11-01,2024-11-26,1584,2024-11-27,2025-11-11,250,2025-11-12,2025-12-10,21
4,4,2018-11-01,2024-12-25,1605,2024-12-26,2025-12-10,250,2025-12-11,2026-01-08,21
5,5,2018-11-01,2025-01-23,1626,2025-01-24,2026-01-08,250,2026-01-09,2026-02-06,21
6,6,2018-11-01,2025-02-21,1647,2025-02-24,2026-02-06,250,2026-02-09,2026-03-09,21
7,7,2018-11-01,2025-03-24,1668,2025-03-25,2026-03-09,250,2026-03-10,2026-04-07,21
8,8,2018-11-01,2025-04-22,1689,2025-04-23,2026-04-07,250,2026-04-08,2026-05-06,21
9,9,2018-11-01,2025-05-21,1710,2025-05-22,2026-05-06,250,2026-05-07,2026-06-04,21


## 2. Multi-step strategy: direct, not recursive

One model per horizon, each mapping features known at *t* straight to the outcome at *t + h*. The recursive alternative feeds a one-step model its own output, so by h = 30 the compounded error dominates. Direct costs one model per horizon and buys an error that does not compound.

In [3]:
from mru_forecast.features.engineering import make_supervised

for h in config["horizons"]:
    d = make_supervised(frame, "USD_MRU", h, config)
    print(f"h={h:>2}  rows={len(d):>5}  features={d.X.shape[1]}  "
          f"target={d.transform}  y std={d.y.std():.5f}")

h= 1  rows= 2052  features=69  target=log_return  y std=0.00453
h= 7  rows= 2046  features=69  target=log_return  y std=0.00739
h=30  rows= 2023  features=69  target=log_return  y std=0.01608


### No leakage — verified, not asserted

Recompute every feature on a truncated history; the shared rows must be identical.

In [4]:
from mru_forecast.features.engineering import build_features

full = build_features(frame, "USD_MRU", config)
cut = frame.index[-60]
truncated = build_features(frame.loc[:cut], "USD_MRU", config)
difference = (full.loc[truncated.index] - truncated).abs().max()
offenders = difference[difference > 1e-12].dropna()
print(f"features that change when the future is removed: {list(offenders.index) or 'none'}")

features that change when the future is removed: none


## 3. Backtest results

Loaded from `scripts/backtest.py`; re-run that script to regenerate.

In [5]:
from mru_forecast.config import resolve_path
from mru_forecast.evaluation.backtesting import BacktestResults

reports = resolve_path("reports", config)
try:
    results = BacktestResults.from_parquet(reports)
    predictions = results.predictions
    print(f"{len(predictions)} predictions over {predictions['origin'].nunique()} origins")
except FileNotFoundError:
    predictions = None
    print("run `python scripts/backtest.py` first")

9072 predictions over 252 origins


In [6]:
from mru_forecast.evaluation.metrics import metrics_table

if predictions is not None:
    point = metrics_table(predictions, group_by=["target", "horizon", "model"], naive_column="naive")
    display(point[["target", "horizon", "model", "mae", "rmse", "mape",
                   "directional_accuracy", "theil_u2", "mae_skill_vs_naive_pct"]].round(4))

,target,horizon,model,mae,rmse,mape,directional_accuracy,theil_u2,mae_skill_vs_naive_pct
0,EUR_MRU,1,drift,0.2017,0.2991,0.4405,48.6486,1.0036,-1.2286
1,EUR_MRU,1,mean_reversion,0.1993,0.2955,0.4353,56.7568,0.9917,-0.0491
2,EUR_MRU,1,naive,0.1992,0.2980,0.4351,NaN,1.0000,0.0000
3,EUR_MRU,1,prophet,0.7320,0.9058,1.6042,55.4054,3.0398,-267.4155
4,EUR_MRU,1,seasonal_naive,0.3785,0.4920,0.8258,47.7477,1.6511,-89.9679
5,EUR_MRU,1,xgboost,0.2177,0.2994,0.4751,55.8559,1.0046,-9.2541
6,EUR_MRU,7,drift,0.4166,0.5485,0.9103,50.3968,1.0316,-2.7256
7,EUR_MRU,7,mean_reversion,0.4028,0.5279,0.8794,53.1746,0.9928,0.6873
8,EUR_MRU,7,naive,0.4056,0.5317,0.8855,NaN,1.0000,0.0000
9,EUR_MRU,7,prophet,0.8543,1.0623,1.8712,59.1270,1.9978,-110.6507


**Theil's U2** is the model's RMSE divided by the random walk's. Below 1 the model beats a random walk; at or above 1 it does not.

`directional_accuracy` is `NaN` for the naive forecast on purpose: predicting no change expresses no direction, so there is nothing to score.

In [7]:
if predictions is not None:
    subset = point[(point["target"] == "USD_MRU") & (point["horizon"] == 1)]
    display(viz.metric_bar_chart(subset, "mae", title="USD/MRU, h=1: MAE (lower is better)"))

In [8]:
if predictions is not None:
    subset = point[(point["target"] == "USD_MRU")]
    display(viz.regime_chart(subset.assign(regime=subset["horizon"].astype(str)),
                             "theil_u2", regime_column="regime",
                             title="USD/MRU: Theil U2 by horizon (1.0 = ties the random walk)"))

## 4. Feature importance

In [9]:
from mru_forecast.pipeline.forecasting_pipeline import ForecastingPipeline

try:
    pipeline = ForecastingPipeline.load(config=config)
    importance = pipeline.feature_importance("USD_MRU", 1, 15)
    display(viz.metric_bar_chart(importance, "gain", group_column="feature",
                                 title="USD/MRU h=1: top features by gain",
                                 value_format=".3f", lower_is_better=False, height=460))
except (FileNotFoundError, KeyError, TypeError) as exc:
    print(f"train first (`python scripts/train.py`): {exc}")